<a href="https://colab.research.google.com/github/Youssif-Kady/flayrank_task1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Rule Logic Overview
Our baseline rule flags pages that generate **high impressions but suffer from poor average positions (page 2 or lower, position > 10)** or **sub-optimal CTR**.

* **Score Calculation:** $Score = Total\_Impressions \times \frac{1}{Avg\_Position + 1}$
* **Action Labels & Reason Codes:**
  1. `OPTIMIZE_TITLE_AND_META` — Reason Code: `HIGH_IMPRESSION_POOR_RANK` (Average position > 10 with significant impression density).
  2. `REFRESH_CONTENT` — Reason Code: `CTR_OPTIMIZATION_CANDIDATE` (Good ranking position <= 10, but low overall CTR).8

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import polars as pl
import pandas as pd
import numpy as np
from google.colab import userdata
from huggingface_hub import HfApi, hf_hub_download, login

# 1. Authenticate & Download March 2026 Dataset
hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token, add_to_git_credential=False)

REPO_ID = "FlyRank/internship-warehouse"
api = HfApi(token=hf_token)
all_files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
march_files = [f for f in all_files if "2026-03" in f and f.endswith(".parquet")]

if not march_files:
    march_files = [f for f in all_files if f.endswith(".parquet")][:5]

local_files = [hf_hub_download(repo_id=REPO_ID, filename=f, repo_type="dataset", token=hf_token) for f in march_files]
df_march = pl.read_parquet(local_files)

# 2. Build Baseline Features & Score
df_rule = df_march.group_by("content_hash_id").agg([
    pl.col("gsc_clicks").sum().alias("total_clicks"),
    pl.col("gsc_impressions").sum().alias("total_impressions"),
    pl.col("gsc_avg_position").mean().alias("avg_position")
]).filter(pl.col("total_impressions") > 0).to_pandas()

# Score & Reason Codes
df_rule['ctr'] = df_rule['total_clicks'] / df_rule['total_impressions']
df_rule['baseline_score'] = (df_rule['total_impressions'] * (1 / (df_rule['avg_position'] + 1))).round(4)
df_rule['reason_code'] = np.where(df_rule['avg_position'] > 10, "HIGH_IMPRESSION_POOR_RANK", "CTR_OPTIMIZATION_CANDIDATE")
df_rule['action_label'] = np.where(df_rule['avg_position'] > 10, "OPTIMIZE_TITLE_AND_META", "REFRESH_CONTENT")

# Sort Ranked Queue
df_queue = df_rule.sort_values(by="baseline_score", ascending=False).reset_index(drop=True)

# Save to CSV
os.makedirs("work/outputs", exist_ok=True)
csv_path = "work/outputs/baseline_action_score.csv"
df_queue[["content_hash_id", "baseline_score", "reason_code", "action_label"]].to_csv(csv_path, index=False)

print(f"✅ Ranked Queue written to {csv_path} (Total Rows: {len(df_queue)})")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


✅ Ranked Queue written to work/outputs/baseline_action_score.csv (Total Rows: 176738)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Top-20 Skeptical Review

1. **Rank 1:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Intent is navigational for a competitor brand where our domain will never satisfy user intent.
2. **Rank 2:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* One-time seasonal traffic spike that dies off next month.
3. **Rank 3:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* SERP feature displays a direct answer card (zero-click query).
4. **Rank 4:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Target URL is an orphaned landing page scheduled for deprecation.
5. **Rank 5:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Outdated technical specification requiring full overhaul rather than simple refresh.
6. **Rank 6:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* High bounce rate due to currency/locale mismatch.
7. **Rank 7:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Internal product team intentionally unlisted page due to stock depletion.
8. **Rank 8:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Page delivers downloadable PDF asset with non-indexable metadata.
9. **Rank 9:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Heavy competitor dominance with AI Overviews taking up top-of-page real estate.
10. **Rank 10:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Query represents ambiguous intent spanning completely different industries.
11. **Rank 11:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Canonical tag issue pointing signal equity to another URL.
12. **Rank 12:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Content already optimized recently, awaiting Google re-indexing cycle.
13. **Rank 13:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Broad term ranking naturally on page 3 without conversion intent.
14. **Rank 14:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Page contains video content where users watch inline rather than clicking.
15. **Rank 15:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Query contains adult or sensitive term filters applied by Search engines.
16. **Rank 16:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Local search intent query where user needs map pack, not web page link.
17. **Rank 17:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Page under active redesign test with broken CSS rendering on mobile.
18. **Rank 18:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Query volume driven by single automated web crawler / bot traffic.
19. **Rank 19:** `OPTIMIZE_TITLE_AND_META` | `HIGH_IMPRESSION_POOR_RANK` — *What would make it wrong:* Historical article covering obsolete product version (e.g., legacy software).
20. **Rank 20:** `REFRESH_CONTENT` | `CTR_OPTIMIZATION_CANDIDATE` — *What would make it wrong:* Page ranking well for transactional query but cart flow is currently broken.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak Picks & Leakage Audit

* **Leakage Verification:** No future-window search signals or target labels (`gsc_clicks` from $t>T-1$) were included in the baseline scoring formula.
* **Weak Picks Identification:** Ranks with high impressions but positions > 50 (e.g., deep page 5 rankings) score artificially high due to impression volume alone, even though moving a page from position 55 to 45 yields zero practical business impact.
* **Model Improvement Direction for W05:** Machine learning models must weight positional thresholds non-linearly to penalize pages sitting beyond position 20.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.